# 03. Supervised Fashion Trend Classification: Random Forest vs. KNN
### AI-Based Clothing Sales Forecasting & Inventory Optimization System

This notebook trains and compares two supervised classification algorithms to categorize fashion products into **`rising`**, **`stable`**, or **`declining`** trends:
1. **Primary Model**: Random Forest Classifier (Ensemble decision trees with Gini feature importance)
2. **Baseline Model**: K-Nearest Neighbors Classifier (Distance-based non-parametric classifier)

#### Evaluation Metrics:
- Accuracy & Macro F1-score
- Multi-class Confusion Matrix
- Gini Feature Importance Ranking

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from src.classification.trend_classifier import FashionTrendModel
from src.classification.baseline_knn import train_and_evaluate_knn

feat_path = '../data/processed/feature_matrix.csv' if os.path.exists('../data/processed/feature_matrix.csv') else 'data/processed/feature_matrix.csv'
df_features = pd.read_csv(feat_path)
print(f"Loaded feature matrix with {df_features.shape[0]:,} records.")

## 1. Train & Benchmark Random Forest vs. KNN Baseline

In [ ]:
# Train Random Forest
rf_model = FashionTrendModel(n_estimators=150, max_depth=10)
rf_report = rf_model.fit_and_evaluate(df_features, test_size=0.2)

# Train KNN Baseline
knn_model, scaler, knn_report = train_and_evaluate_knn(df_features, n_neighbors=5, test_size=0.2)

print("Random Forest Test Accuracy:", rf_report['test_metrics']['accuracy'], "| Macro F1:", rf_report['test_metrics']['f1_macro'])
print("KNN Baseline Test Accuracy:", knn_report['test_metrics']['accuracy'], "| Macro F1:", knn_report['test_metrics']['f1_macro'])

## 2. Multi-Class Confusion Matrix Visualization

In [ ]:
labels = rf_report['test_metrics']['labels']
cm_rf = np.array(rf_report['test_metrics']['confusion_matrix'])

plt.figure(figsize=(7, 5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Purples', xticklabels=labels, yticklabels=labels)
plt.title('Random Forest Trend Classification Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('Actual Trend Status')
plt.xlabel('Predicted Trend Status')
plt.tight_layout()
plt.show()

## 3. Gini Feature Importance Ranking

In [ ]:
df_imp = pd.DataFrame(rf_report['feature_importance']).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_imp, x='importance', y='feature', palette='mako')
plt.title('Top 10 Most Predictive Features for Fashion Trend Direction', fontsize=13, fontweight='bold')
plt.xlabel('Mean Gini Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()